# A Socratic Journey Through Neural Network Quantization

### From a single neuron to precision and bit-budget choices

This notebook is for readers who know basic Python and are new to quantization. You do not need to know ResNet internals or quantization jargon in advance. We start with what a layer computes, then ask how changing its weights affects predictions.

## How to use this notebook

At each stop, pause at **Predict** and write a guess before running the experiment. Use the printed measurements or plot as evidence, then finish with **Explain** in your own words. Expand the short answer hints only after attempting the question.

Our central question is: **How can we store weights with fewer bits while keeping the model's useful behavior?**

## Route

1. Build a mental model: inputs, weights, layers, predictions, and what quantization changes.
2. Set a baseline by evaluating the same pretrained model before modifying it.
3. Find the failure by comparing one global scale at 8 bits and 4 bits.
4. Improve the representation with channel and group scales, clipping, and zero-points.
5. Spend bits where useful with measured layer sensitivity and mixed precision.

> **Scope:** These are post-training, weight-only experiments on a pretrained ResNet-18 and a fixed 500-image, 10-class Imagenette sample. Results depend on this model, sample, seed, and implementation. Some code illustrates mechanisms rather than measuring deployment latency or packed-integer storage.


In [ ]:
import sys
from pathlib import Path
import copy
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Ensure src/ is on sys.path
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(ROOT / "src"))

from models.vision import load_pretrained_model
from data.imagenette import make_test_loader, make_calibration_loader
from evaluation.metrics import evaluate
from quantization.integer.weight import (
    quantize_weights,
    quantize_per_channel,
    quantize_groupwise,
    compute_layer_sensitivities,
    quantize_sensitivity_guided,
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Stop 1 ? What are we changing when we quantize?

### From one neuron to a network

A neuron multiplies input values by learned weights, adds a bias, and applies an activation. A layer is often written as $h=Wx+b$ and $y=\mathrm{ReLU}(h)=\max(0,h)$. A network chains layers. During training the weights are adjusted so the final class scores match labels; at inference, fixed weights map an image to scores and the largest score gives a prediction.

### What quantization changes

Weight quantization replaces each floating-point weight $w$ with a nearby value $\hat w$ from a small grid. A layer then uses $\hat W$ instead of $W$. This introduces an approximation to learned parameters; the important question is whether that approximation changes layer outputs enough to alter final predictions.

For a symmetric signed $b$-bit grid, one common rule is

$$s=\frac{\max |W|}{2^{b-1}-1},\quad q=\mathrm{clip}(\mathrm{round}(W/s),-2^{b-1},2^{b-1}-1),\quad \hat W=qs.$$

The scale $s$ is the spacing between represented values. Fewer bits mean fewer choices, but the way we choose the scale also matters.

**Predict:** If weights in a layer become smaller or zero, what might happen to that layer's output? Would every layer be affected equally?

The next cell is a toy illustration of repeated gain changes. It is not a ResNet or accuracy experiment: it isolates signal magnitude to build intuition.

<details><summary>Answer hint</summary>

Smaller effective weights can shrink outputs and larger ones can grow them. Repeating changes can accumulate. Real networks include residual paths, normalization, nonlinearities, and data-dependent activations, so the toy pattern is not a universal law about accuracy or neuron death.

</details>


In [ ]:
# Toy illustration: repeated layer gain changes
# This isolates signal norms; it is not a model accuracy or ResNet simulation.


def simulate_propagation():
    L = 18
    dim = 256
    torch.manual_seed(42)
    x0 = torch.randn(1, dim)

    # Condition 1: Balanced FP32 (gain g ~ 1.0)
    norm_fp32 = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5)
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_fp32.append(x.norm().item())

    # Condition 2: Coarse Quantization / Over-Pruned (75% weights zeroed -> g ~ 0.55)
    norm_zero = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5)
        mask = (torch.rand_like(w) > 0.75).float()
        w = w * mask
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_zero.append(x.norm().item())

    # Condition 3: Gain Mismatch / Amplification (gain g ~ 1.75)
    norm_blowup = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5) * 1.75
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_blowup.append(x.norm().item())

    plt.figure(figsize=(10, 4.5))
    plt.plot(range(L + 1), norm_fp32, label="Balanced FP32 (Stable Variance)", color="#2ecc71", linewidth=2.5)
    plt.plot(range(L + 1), norm_zero, label="Signal Vanishing (Weights drowned in zero-bin)", color="#3498db", linewidth=2.5)
    plt.plot(range(L + 1), norm_blowup, label="Signal Explosion (Mismatched scale amplification)", color="#e74c3c", linewidth=2.5)

    plt.yscale("log")
    plt.xlabel("Layer Depth ($l = 1 \dots 18$)", fontsize=11)
    plt.ylabel("Activation Vector Norm $\|x_l\|$ (Log Scale)", fontsize=11)
    plt.title("The Physics of Deep Signal Propagation: Why Quantization Stability Matters", fontsize=12, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

simulate_propagation()

### Explain the toy plot

Which curve stays in a similar range, and which shrinks or grows? What does this suggest about approximation errors repeated across layers? The plot motivates checking outputs; it does not predict model accuracy by itself.


## Stop 2 ? Establish a fair reference

The setup cell loads a pretrained model and a deterministic image sample; it does not train the model. We will compare copies of this same model on the same images and preprocessing.

**Predict:** Should we expect 100% accuracy if the original model already makes mistakes? What must remain fixed for a useful comparison?

<details><summary>Answer hint</summary>

No. Compare each quantized copy to the same FP32 model on the same data and preprocessing. The measured difference describes this evaluation setup, not a universal property of the method.

</details>


In [ ]:
torch.set_num_threads(4)
device = torch.device("cpu")
checkpoint_path = ROOT / "results/checkpoints/resnet18_pretrained_fp32.pt"

# Load torchvision pretrained ResNet-18
model, weights = load_pretrained_model("resnet18", checkpoint_path)
model = model.to(device).eval()

# Load 500-sample Imagenette test split and calibration split
test_loader, class_indices = make_test_loader(
    ROOT / "data/raw/imagenette2-160",
    weights,
    batch_size=32,
    images_per_class=50,
    seed=42,
)

calibration_loader = make_calibration_loader(
    ROOT / "data/raw/imagenette2-160",
    weights,
    batch_size=32,
    max_images=128,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Pretrained ResNet-18: {total_params:,} parameters ({total_params*4/1e6:.2f} MB in FP32)")
print(f"Validation Dataset: Imagenette ({len(test_loader.dataset)} images across 10 classes)")

## Stop 3 ? Does one global scale work? (8 bits, then 4)

**Predict:** An 8-bit grid has more levels than a 4-bit grid. What do you expect as we reduce the number of levels? Why might one scale for every layer be a problem?

?Global? here means one shared symmetric scale for all quantized weights: a deliberately simple baseline, not the only way to quantize a network. First record FP32 and INT8. Before the INT4 experiment, write down your prediction.


In [ ]:
# 1. Baseline FP32
fp32_res = evaluate(model, test_loader, device, class_indices)
print(f"Baseline FP32 Accuracy: {fp32_res['accuracy']*100:.2f}% | Loss: {fp32_res['loss']:.4f}")

# 2. Global INT8
m_int8 = copy.deepcopy(model)
quantize_weights(m_int8, bits=8)
int8_res = evaluate(m_int8, test_loader, device, class_indices)
print(f"Global INT8 Accuracy:   {int8_res['accuracy']*100:.2f}% | Loss: {int8_res['loss']:.4f}")
print(f"Storage reduction:      46.76 MB -> 11.72 MB (4x compression!)")

### Before the 4-bit experiment

Compare FP32 and INT8 accuracy and loss. Was the change large on this sample? Predict whether reducing to 4 bits will cause a gradual decline or a sharp failure.


In [ ]:
m_int4 = copy.deepcopy(model)
meta_int4 = quantize_weights(m_int4, bits=4)
int4_res = evaluate(m_int4, test_loader, device, class_indices)
print(f"Global INT4 Accuracy: {int4_res['accuracy']*100:.2f}% (Complete Collapse!)")

# Inspect weight distributions:
conv1_w = dict(model.named_modules())["conv1"].weight.data
l4_w = dict(model.named_modules())["layer4.1.conv2"].weight.data

scale = meta_int4["scale"]
zero_thresh = scale / 2.0

print(f"\nDiagnostic Analysis:")
print(f"-> Global Scale (set by conv1 max): {scale:.4f}")
print(f"-> Zero-bin threshold [-thresh, +thresh]: [-{zero_thresh:.4f}, +{zero_thresh:.4f}]")
print(f"-> conv1 max magnitude:                  {conv1_w.abs().max():.4f}")
print(f"-> layer4.1.conv2 mean magnitude:        {l4_w.abs().mean():.4f}")

In [ ]:
# Measure percentage of weights turned to EXACT ZERO in each layer under Global INT4
layer_names = []
zero_percentages = []

for name, module in m_int4.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        w_q = module.weight.data
        zero_pct = (w_q == 0.0).float().mean().item() * 100.0
        layer_names.append(name.replace("layer", "L").replace(".conv", ".c"))
        zero_percentages.append(zero_pct)

plt.figure(figsize=(12, 4))
plt.bar(range(len(layer_names)), zero_percentages, color="#e74c3c", edgecolor="black")
plt.xticks(range(len(layer_names)), layer_names, rotation=60, ha="right", fontsize=9)
plt.ylabel("% Weights Mapped to 0.0", fontsize=11)
plt.title("The Outlier Tyranny: 99.9% of Deep Layer Weights Rounded to 0.0 under Global INT4", fontsize=12, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Explain the INT4 evidence

Connect the shared scale and zero-bin to the layer-wise zero percentages. **In one sentence:** What is the failure mode of one global scale at low bit width in this example? The zero count supports a mechanism but does not alone prove why accuracy changed; weight locations and downstream computation matter too.


## Stop 4 ? Give different regions different grids

The global-scale experiment suggests that layers or channels can have different ranges. **Predict:** What might happen if each output channel chooses its own scale? What cost or limitation could finer scale selection introduce?

We compare per-channel quantization with and without clipping. Per-channel scaling changes scale granularity. Clipping changes the represented range: it can reduce rounding error for common values while increasing error for values outside the range. These are separate choices. The OMSE option searches clipping candidates to reduce weight reconstruction error; that objective does not guarantee a proportional accuracy gain.


In [ ]:
# 1. Per-Channel INT4 Standard
m_pc = copy.deepcopy(model)
quantize_per_channel(m_pc, bits=4, mse_clipping=False)
pc_res = evaluate(m_pc, test_loader, device, class_indices)
print(f"Per-Channel INT4 (PyTorch Standard): {pc_res['accuracy']*100:.2f}% (Jump from 10.0% -> 89.6%!)")

# 2. Per-Channel INT4 + OMSE Clipping
m_omse = copy.deepcopy(model)
quantize_per_channel(m_omse, bits=4, mse_clipping=True)
omse_res = evaluate(m_omse, test_loader, device, class_indices)
print(f"Per-Channel INT4 + OMSE Clipping:    {omse_res['accuracy']*100:.2f}% (Gained another +3.4%!)")

### Checkpoint: did finer scale granularity help?

Compare both per-channel results with global INT4. Did clipping help here, and what did it trade away? These measured differences are specific to this evaluation sample.


## Stop 5 ? Is a channel still too large a region?

Per-channel scaling gives each output channel its own range, but a channel can still contain weights with different local ranges. **Predict:** Would smaller groups fit the values more closely? What trade-off might come with more scales?

We compare groups of 32 contiguous weights and two grid choices: **symmetric**, centered at zero, and **asymmetric**, shifted with a zero-point to cover a non-centered range. These experiments illustrate group-wise quantization; they are not equivalent to every configuration in AWQ, GPTQ, or GGUF.


In [ ]:
# Group-wise INT4 (G=32)
m_gw4_sym = copy.deepcopy(model)
quantize_groupwise(m_gw4_sym, bits=4, group_size=32, asymmetric=False)
gw4_sym = evaluate(m_gw4_sym, test_loader, device, class_indices)

m_gw4_asym = copy.deepcopy(model)
quantize_groupwise(m_gw4_asym, bits=4, group_size=32, asymmetric=True)
gw4_asym = evaluate(m_gw4_asym, test_loader, device, class_indices)

# Group-wise INT3 (G=32)
m_gw3_sym = copy.deepcopy(model)
quantize_groupwise(m_gw3_sym, bits=3, group_size=32, asymmetric=False)
gw3_sym = evaluate(m_gw3_sym, test_loader, device, class_indices)

m_gw3_asym = copy.deepcopy(model)
quantize_groupwise(m_gw3_asym, bits=3, group_size=32, asymmetric=True)
gw3_asym = evaluate(m_gw3_asym, test_loader, device, class_indices)

print(f"Group-wise INT4 (G=32, Symmetric):  {gw4_sym['accuracy']*100:.2f}%")
print(f"Group-wise INT4 (G=32, Asymmetric): {gw4_asym['accuracy']*100:.2f}% (Only 0.8% drop from FP32!)")
print(f"Group-wise INT3 (G=32, Symmetric):  {gw3_sym['accuracy']*100:.2f}%")
print(f"Group-wise INT3 (G=32, Asymmetric): {gw3_asym['accuracy']*100:.2f}% (+9.2% jump from zero-points!)")

### Checkpoint: compare the grids

Compare symmetric and asymmetric results at both bit widths. Does the zero-point help in every comparison? Its benefit depends on the weight distribution and quantizer details.


## Stop 6 ? Should every layer get the same bit width?

Layers differ both in parameter count and in how much quantization perturbs model outputs. **Predict:** If one small layer is sensitive but another layer contains many weights and tolerates coarser quantization, where should we spend the next bit?

The chart shows parameter counts, not sensitivity. The next cell estimates sensitivity using calibration examples, then allocates candidate bit widths under a target average weight-bit budget. Effective weight bits are an average; scale metadata and other model state add storage, and file size depends on packing and serialization.


In [ ]:
modules = [(name, m) for name, m in model.named_modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
total_w = sum(m.weight.numel() for _, m in modules)

counts = {
    "conv1 & layer1": sum(m.weight.numel() for name, m in modules if "conv1" in name or "layer1" in name),
    "layer2": sum(m.weight.numel() for name, m in modules if "layer2" in name),
    "layer3": sum(m.weight.numel() for name, m in modules if "layer3" in name),
    "layer4 (Bulk)": sum(m.weight.numel() for name, m in modules if "layer4" in name),
    "fc (Classifier)": sum(m.weight.numel() for name, m in modules if "fc" in name),
}

plt.figure(figsize=(7, 7))
plt.pie(
    counts.values(),
    labels=[f"{k}\n({v:,} | {v/total_w*100:.1f}%)" for k, v in counts.items()],
    autopct="%1.1f%%",
    startangle=140,
    colors=["#3498db", "#2ecc71", "#f39c12", "#e74c3c", "#9b59b6"],
    explode=[0.05, 0.05, 0.05, 0.08, 0.05],
)
plt.title("ResNet-18 Parameter Distribution: layer4 Contains 75.1% of All Weights!", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Checkpoint: count is not sensitivity

The chart tells us where storage is spent; sensitivity estimates tell us which layer perturbations matter under this procedure. Why do we need both to choose a bit allocation?


In [ ]:
print("Computing layer-wise output perturbation sensitivities on calibration data...")
sens = compute_layer_sensitivities(
    model, calibration_loader, device,
    candidate_bits=(1, 2, 3, 4, 8),
    group_size=32,
    asymmetric=True,
    max_images=128,
)

budgets = [2.0, 2.2, 2.5, 3.0, 3.5, 4.0]
pareto_results = []

for b in budgets:
    m_sg = copy.deepcopy(model)
    meta = quantize_sensitivity_guided(
        m_sg, target_effective_bits=b, sensitivities=sens, group_size=32, asymmetric=True
    )
    res = evaluate(m_sg, test_loader, device, class_indices)
    actual_b = meta["effective_weight_bits"]
    pareto_results.append((actual_b, res["accuracy"] * 100))
    print(f"Target Budget {b:.1f}b -> Actual: {actual_b:.2f}b | Accuracy: {res['accuracy']*100:.2f}% | Loss: {res['loss']:.4f}")

In [ ]:
bits_vals, acc_vals = zip(*pareto_results)

plt.figure(figsize=(10, 5))
plt.axhline(fp32_res["accuracy"] * 100, color="gray", linestyle="--", label=f"FP32 Baseline ({fp32_res['accuracy']*100:.1f}%)")

# Baselines for comparison
plt.scatter([1.0, 2.0, 4.0], [10.0, 10.0, 10.0], color="red", marker="x", s=80, label="Blind Global (INT1, INT2, INT4)")
plt.scatter([4.0], [pc_res["accuracy"]*100], color="orange", marker="^", s=80, label="Per-Channel INT4 (89.6%)")
plt.scatter([4.0], [gw4_asym["accuracy"]*100], color="purple", marker="s", s=80, label="Group-wise INT4 (96.6%)")
plt.scatter([3.0], [gw3_asym["accuracy"]*100], color="brown", marker="d", s=80, label="Group-wise INT3 (91.8%)")

# Pareto curve
plt.plot(bits_vals, acc_vals, color="#27ae60", marker="o", linewidth=2.5, markersize=8, label="Sensitivity-Guided Pareto Frontier")

for b, a in zip(bits_vals, acc_vals):
    plt.annotate(f"{a:.1f}% ({b:.2f}b)", (b, a), textcoords="offset points", xytext=(-10, 8), fontweight="bold", color="#196f3d")

plt.xlabel("Effective Weight Bits", fontsize=11)
plt.ylabel("Top-1 Accuracy (%)", fontsize=11)
plt.title("The Quantization Pareto Frontier: Maximizing Accuracy with Minimal Effective Bits", fontsize=12, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Read the Pareto curve

Each point pairs effective weight bits with measured accuracy. A point is attractive when no other point uses fewer bits and achieves at least as much accuracy. Which point would you choose if accuracy close to FP32 mattered most? Which if storage were the main constraint? Name the constraint driving your choice.


## Stop 7 ? Explore a bit budget

Before running the custom budget, predict which layers will receive more precision and how accuracy might move. Then inspect the allocation.

**Reflect:** Does this establish a universal optimal allocation? Consider what depends on this model, calibration sample, candidate bit widths, and allocator objective. The printed storage estimate is approximate and is not a complete serialized model or runtime memory measurement.


In [ ]:
def test_custom_budget(target_bits=2.5):
    m_custom = copy.deepcopy(model)
    meta = quantize_sensitivity_guided(
        m_custom, target_effective_bits=target_bits, sensitivities=sens, group_size=32, asymmetric=True
    )
    res = evaluate(m_custom, test_loader, device, class_indices)
    print(f"=== Budget: {target_bits:.2f} bits ===")
    print(f"Achieved Effective Bits: {meta['effective_weight_bits']:.2f}")
    print(f"Accuracy:                {res['accuracy']*100:.2f}% (FP32 baseline is {fp32_res['accuracy']*100:.2f}%)")
    print(f"Estimated Storage:       {meta['scale_count']*5/1e6 + meta['quantized_weight_count']*meta['effective_weight_bits']/8/1e6:.2f} MB")
    print("\nLayer Precision Allocation:")
    for name, d in meta["layers"].items():
        print(f"  {name:30s} -> {d['bits']} bits ({d['weights']:,} weights)")

# Try a budget of 2.3 bits:
test_custom_budget(2.3)